# 64-Neuron 4-Class — Zero-M (NuMIT) Z-Score Analysis

Replaces the broken time-shuffle null with the proven **Zero-M Gaussian VAR(1) null** methodology from the `Z - scoring issue/` diagnostics package.

## Why the Time-Shuffle Null Failed

The time-shuffle (independent per-neuron permutation) produces near-identical W values to observed data because the Gaussian copula's `double_union` optimizer is sensitive to marginal structure — not temporal ordering. The null standard deviation collapses to ~10⁻⁶, inflating z-scores to ~10⁶ and making them meaningless. At k ≥ 8, covariance ill-conditioning causes NaN failures.

## The Zero-M (NuMIT) Solution

1. **Gaussian copula normalization** — rank-based inverse-normal transform
2. **Multi-strategy robust W/M** — tries Adam/Newton optimizers, different lags/strides, ridge regularization
3. **Zero-M null** — builds a VAR(1) model matching observed TDMI but with M=0 by construction
4. **Proper Z-scoring** — compares observed M against the Zero-M null with full p-values

Reference: `Z - scoring issue/run_z_scoring_diagnostics.py`

## 1. Imports & Path Setup

In [2]:
import sys
sys.path.insert(0, r"C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files")

import hashlib
import io
import itertools
import json
import math
import time as time_module
import warnings
from contextlib import redirect_stdout
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.linalg import solve_discrete_lyapunov
from scipy.optimize import root_scalar
from scipy.stats import norm, rankdata, wishart
from IPython.display import display

import importlib
import seeded_runs_common as seeded_runs_common
seeded_runs_common = importlib.reload(seeded_runs_common)

# ── Paths ──────────────────────────────────────────────────────────
CHECKPOINT_ROOT = seeded_runs_common.CHECKPOINT_ROOT
DEVICE = seeded_runs_common.DEVICE
TASKS = seeded_runs_common.TASKS
build_seeded_pair = seeded_runs_common.build_seeded_pair
get_pair_checkpoint_paths = seeded_runs_common.get_pair_checkpoint_paths
load_default_caches = seeded_runs_common.load_default_caches

RUN_LABEL = "seeded_run_1_64n"
TASK_KEY = "4class"
MEM_DISTRIBUTION_FAMILY = "lognormal"
RUN_DIR = CHECKPOINT_ROOT / RUN_LABEL / f"{TASK_KEY}_{MEM_DISTRIBUTION_FAMILY}"
OUTPUT_DIR = RUN_DIR / "zero_m_zscores"
OUTPUT_DIR.mkdir(exist_ok=True)

# Set nb_recurrent to 64 for model building
seeded_runs_common.BASE_PRMS["nb_recurrent"] = 64

N_NULL = 20
MAX_ATTEMPTS = 900

print(f"Device: {DEVICE}")
print(f"Run directory: {RUN_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"N_NULL={N_NULL}, MAX_ATTEMPTS={MAX_ATTEMPTS}")

Device: cuda
Run directory: C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files\Checkpoints\SeededRuns\seeded_run_1_64n\4class_lognormal
Output directory: C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files\Checkpoints\SeededRuns\seeded_run_1_64n\4class_lognormal\zero_m_zscores
N_NULL=20, MAX_ATTEMPTS=900


## 2. W/M Estimation Engine (Gaussian Copula + Robust Fallback)

Functions ported from `Z - scoring issue/run_z_scoring_diagnostics.py`. The key innovations over the naive approach:

1. **Gaussian copula normalization** — rank-based inverse-normal transform handles binary spike data
2. **Multi-strategy fallback** — tries Adam/Newton optimizers, different lags, strides, and ridge values
3. **Zero-M null generator** — builds VAR(1) surrogates with M=0 by construction

In [3]:
# ═══════════════════════════════════════════════════════════════════
#  Core functions ported from Z - scoring issue/run_z_scoring_diagnostics.py
# ═══════════════════════════════════════════════════════════════════

def _stable_seed(*parts):
    """Deterministic hashing seed."""
    token = "|".join(map(str, parts)).encode("utf-8")
    return int(hashlib.sha256(token).hexdigest()[:8], 16)


def _gc_normalize(data):
    """Gaussian copula: rank → uniform → inverse normal."""
    transformed = np.zeros_like(data, dtype=np.float64)
    for idx, row in enumerate(data):
        if np.allclose(row, row[0]):
            continue
        ranks = rankdata(row, method="average")
        uniform = np.clip((ranks - 0.5) / len(row), 1e-6, 1.0 - 1e-6)
        transformed[idx] = norm.ppf(uniform)
    return transformed


def _inject_jitter(data, eps=1e-6):
    """Handle degenerate (constant) rows."""
    row_std = np.nanstd(data, axis=1)
    deg_idx = np.flatnonzero(row_std <= 1e-12)
    if deg_idx.size == 0:
        return data
    data = data.copy()
    base = np.linspace(-1.0, 1.0, data.shape[1], dtype=np.float64)
    for offset, ridx in enumerate(deg_idx, start=1):
        data[ridx] = (eps * offset) * base
    return data


def _regularize_cov(cov, ridge=1e-9):
    """Regularize covariance matrix."""
    cov = np.asarray(cov, dtype=np.float64)
    cov = np.nan_to_num(cov, nan=0.0, posinf=0.0, neginf=0.0)
    cov = 0.5 * (cov + cov.T)
    scale = np.trace(cov) / max(cov.shape[0], 1)
    if (not np.isfinite(scale)) or scale <= 0.0:
        scale = 1.0
    return cov + float(ridge) * scale * np.eye(cov.shape[0], dtype=np.float64)


def _logdet_pd(mat):
    """Safe log-determinant for positive definite matrix."""
    sign, val = np.linalg.slogdet(mat)
    if sign <= 0 or (not np.isfinite(val)):
        raise ValueError("Matrix not positive definite for logdet.")
    return float(val)


def _spectral_radius(A):
    """Spectral radius (max absolute eigenvalue)."""
    eig = np.linalg.eigvals(A)
    return float(np.max(np.abs(eig)))


def _sigmoid(x, M=1.0):
    x = np.clip(float(x), -60.0, 60.0)
    return float(M) / (1.0 + np.exp(-x))


def _mi_var1_bits(A, V):
    """Mutual information (bits) of a VAR(1) process."""
    S = solve_discrete_lyapunov(A, V)
    S = _regularize_cov(S, ridge=1e-12)
    return 0.5 * (_logdet_pd(S) - _logdet_pd(V)) / np.log(2.0)


def _lagged_cov_from_var1(A, V):
    """Lagged covariance from VAR(1) matrices."""
    S = solve_discrete_lyapunov(A, V)
    S = _regularize_cov(S, ridge=1e-12)
    cpf = S @ A.T
    return np.block([[S, cpf], [cpf.T, S]])


def _numit_var1_from_mi(mi_target_bits, nvar, rng, g_cap=0.9995, tol_bits=1e-3):
    """Fit VAR(1) model matching target MI with M=0."""
    A0 = rng.normal(0.0, 1.0, size=(int(nvar), int(nvar)))
    sr = _spectral_radius(A0)
    if (not np.isfinite(sr)) or sr <= 1e-12:
        return None, None, True, np.nan
    A_unit = A0 / sr

    V = wishart.rvs(df=int(nvar) + 1, scale=np.eye(int(nvar)), random_state=rng)
    V = _regularize_cov(V, ridge=1e-9)

    def f_x(x):
        g = float(g_cap) * _sigmoid(x, 1.0)
        A = g * A_unit
        try:
            mi_val = _mi_var1_bits(A, V)
        except Exception:
            return np.nan
        return float(mi_val - float(mi_target_bits))

    xs = np.array([-12, -8, -5, -3, -2, -1, 0, 1, 2, 3, 5, 8, 12], dtype=np.float64)
    fs = np.asarray([f_x(x) for x in xs], dtype=np.float64)

    x_star = None
    near_idx = np.where(np.isfinite(fs) & (np.abs(fs) <= float(tol_bits)))[0]
    if near_idx.size > 0:
        x_star = float(xs[int(near_idx[0])])
    else:
        for idx in range(len(xs) - 1):
            f1, f2 = fs[idx], fs[idx + 1]
            if not (np.isfinite(f1) and np.isfinite(f2)):
                continue
            if f1 * f2 > 0:
                continue
            try:
                sol = root_scalar(f_x, bracket=[float(xs[idx]), float(xs[idx + 1])],
                                  method="brentq", xtol=1e-6, rtol=1e-6, maxiter=100)
            except Exception:
                sol = None
            if sol is not None and sol.converged and np.isfinite(sol.root):
                x_star = float(sol.root)
                break

    if x_star is None:
        return None, None, True, np.nan

    g_star = float(g_cap) * _sigmoid(x_star, 1.0)
    A = g_star * A_unit

    try:
        mi_check = _mi_var1_bits(A, V)
    except Exception:
        return None, None, True, np.nan

    if not np.isfinite(mi_check):
        return None, None, True, np.nan

    allowed_err = max(float(tol_bits), 0.02 * max(float(mi_target_bits), 1e-8))
    if abs(float(mi_check) - float(mi_target_bits)) > allowed_err:
        return None, None, True, float(mi_check)

    return A, V, False, float(mi_check)


def _wm_from_lagged_cov(lagged_cov, ridge=1e-2):
    """W/M from lagged covariance using double_union."""
    from wimfo.gaussian.double_union_gauss import double_union
    from wimfo.utils.utils_gauss import tdmi_from_cov

    cov = _regularize_cov(lagged_cov, ridge=float(ridge))
    nvar = cov.shape[0] // 2

    if nvar > 15:
        solver_candidates = [
            ("Adam", {"atol": 1e-3, "rtol": 1e-3, "max_iter": 400, "window_size": 21}),
            ("Newton", {"max_iter": 40}),
        ]
    else:
        solver_candidates = [
            ("Newton", {"max_iter": 75}),
            ("Adam", {"atol": 1e-3, "rtol": 1e-3, "max_iter": 800, "window_size": 21}),
        ]

    for optimiser, options in solver_candidates:
        with io.StringIO() as buf, redirect_stdout(buf):
            w_nats = double_union(cov, nx=nvar, optimiser=optimiser,
                                  options=options, verbose=False, switch_opt=False)
        if np.isfinite(w_nats):
            tdmi_bits = float(tdmi_from_cov(cov, xdim=nvar))
            w_bits = float(w_nats / np.log(2.0))
            m_bits = float(tdmi_bits - w_bits)
            return w_bits, m_bits

    raise RuntimeError(f"Null W solver failed for nvar={nvar}")


def compute_wm_from_spike_matrix(hidden_data, lag=1, ridge=1e-2):
    """Robust W/M estimation from spike matrix with multi-strategy fallback."""
    from wimfo.W_M_Info import W_M_calculator
    from wimfo.utils.utils_gauss import get_cov

    gdata = _gc_normalize(hidden_data.astype(np.float64).copy())
    gdata = np.nan_to_num(gdata, nan=0.0, posinf=0.0, neginf=0.0)
    gdata = _inject_jitter(gdata)

    opt_order = [
        ("Adam", {"atol": 1e-3, "rtol": 1e-3, "max_iter": 30000}),
        ("Adam", {"atol": 5e-3, "rtol": 5e-3, "max_iter": 60000}),
        ("Newton", None),
    ]
    lag0 = max(int(lag), 1)
    lag_candidates = sorted({lag0, lag0 * 2, lag0 * 4})
    stride_candidates = [1, 2, 4]
    ridge_candidates = sorted({float(ridge), 5.0 * float(ridge), 1e-1, 2e-1, 5e-1, 1.0})

    # Strategy 1: Direct W_M_calculator on data
    for stride in stride_candidates:
        dv = gdata[:, ::stride]
        if dv.shape[1] <= max(8, 2 * dv.shape[0] + 2):
            continue
        for lag_try in lag_candidates:
            for optimiser, options in opt_order:
                try:
                    with io.StringIO() as buf, redirect_stdout(buf):
                        w_bits, m_bits = W_M_calculator(
                            dv, t=lag_try, option="data", type="gaussian",
                            unit="bits", verbose=False,
                            optimiser=optimiser, options=options)
                except Exception:
                    continue
                if np.isfinite(w_bits) and np.isfinite(m_bits):
                    return float(w_bits), float(m_bits), int(dv.shape[1])

    # Strategy 2: Regularized covariance + W_M_calculator
    last_err = None
    for stride in stride_candidates:
        dv = gdata[:, ::stride]
        if dv.shape[1] <= max(8, 2 * dv.shape[0] + 2):
            continue
        for lag_try in lag_candidates:
            try:
                cov = np.asarray(get_cov(dv, t=lag_try), dtype=np.float64)
            except Exception as exc:
                last_err = exc
                continue
            cov = np.nan_to_num(cov, nan=0.0, posinf=0.0, neginf=0.0)
            cov = 0.5 * (cov + cov.T)
            scale = np.trace(cov) / max(cov.shape[0], 1)
            if not np.isfinite(scale) or scale <= 0.0:
                scale = 1.0
            eye = np.eye(cov.shape[0], dtype=np.float64)
            for ridge_try in ridge_candidates:
                lc = cov + ridge_try * scale * eye
                try:
                    evals, evecs = np.linalg.eigh(lc)
                    evals = np.clip(evals, max(scale * 1e-8, 1e-10), None)
                    lc = (evecs * evals[np.newaxis, :]) @ evecs.T
                    lc = 0.5 * (lc + lc.T)
                except np.linalg.LinAlgError as exc:
                    last_err = exc
                    continue
                for optimiser, options in opt_order:
                    try:
                        with io.StringIO() as buf, redirect_stdout(buf):
                            w_bits, m_bits = W_M_calculator(
                                lc, option="distr", type="gaussian",
                                unit="bits", verbose=False,
                                optimiser=optimiser, options=options)
                    except Exception as exc:
                        last_err = exc
                        continue
                    if np.isfinite(w_bits) and np.isfinite(m_bits):
                        return float(w_bits), float(m_bits), int(dv.shape[1])

    raise RuntimeError(f"All W/M estimation paths failed. Last error: {last_err}")


def build_numit_null_distribution(nvar, tdmi_target_bits, n_null=20, seed=12345,
                                   ridge=1e-2, max_attempts=900):
    """Generate Zero-M null distribution matching observed TDMI."""
    rng = np.random.default_rng(int(seed))
    null_m = []
    null_w = []
    model_mi_vals = []
    wm_tdmi_vals = []

    attempts = 0
    while len(null_m) < int(n_null) and attempts < int(max_attempts):
        attempts += 1

        A, V, failed, mi_model = _numit_var1_from_mi(tdmi_target_bits, nvar, rng)
        if failed:
            continue

        try:
            lag_cov = _lagged_cov_from_var1(A, V)
            w_bits, m_bits = _wm_from_lagged_cov(lag_cov, ridge=ridge)
        except Exception:
            continue

        if not (np.isfinite(w_bits) and np.isfinite(m_bits)):
            continue

        null_w.append(float(w_bits))
        null_m.append(float(m_bits))
        model_mi_vals.append(float(mi_model))
        wm_tdmi_vals.append(float(w_bits + m_bits))

    return {
        "null_M_values": null_m,
        "null_W_values": null_w,
        "model_mi_bits": model_mi_vals,
        "wm_tdmi_bits": wm_tdmi_vals,
        "n_null_valid": int(len(null_m)),
        "n_attempts": int(attempts),
    }


def score_observed_vs_null_m(observed_m, null_m_values):
    """Z-score and p-values for observed M vs null distribution."""
    vals = np.asarray(null_m_values, dtype=np.float64)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return {"null_mean": np.nan, "null_std": np.nan, "z": np.nan,
                "p_upper": np.nan, "p_lower": np.nan, "p_two_sided": np.nan}

    mu = float(vals.mean())
    sd = float(vals.std(ddof=1)) if vals.size > 1 else np.nan
    z = float((float(observed_m) - mu) / sd) if (np.isfinite(sd) and sd > 1e-12) else np.nan

    ge = np.sum(vals >= float(observed_m))
    le = np.sum(vals <= float(observed_m))
    p_upper = float((1.0 + ge) / (vals.size + 1.0))
    p_lower = float((1.0 + le) / (vals.size + 1.0))
    p_two = float(min(1.0, 2.0 * min(p_upper, p_lower)))

    return {"null_mean": mu, "null_std": sd, "z": z,
            "p_upper": p_upper, "p_lower": p_lower, "p_two_sided": p_two}


def sample_random_subsets(n_neurons, subset_size, sample_size, seed):
    """Sample random subsets of neurons."""
    total_possible = math.comb(int(n_neurons), int(subset_size))
    target_size = min(int(sample_size), int(total_possible))

    if target_size == total_possible:
        all_subsets = itertools.combinations(range(int(n_neurons)), int(subset_size))
        return [tuple(idx) for idx in all_subsets], total_possible

    rng = np.random.default_rng(int(seed))
    sampled = set()
    while len(sampled) < target_size:
        subset = tuple(sorted(rng.choice(int(n_neurons), size=int(subset_size),
                                         replace=False).tolist()))
        sampled.add(subset)
    return sorted(sampled), total_possible


print("✓ Core W/M estimation engine loaded.")

✓ Core W/M estimation engine loaded.


## 3. Extract Hidden Traces from Checkpoints

In [4]:
def extract_hidden_traces(model, prms, data_cache, n_batches=8, layer_idx=0):
    """Extract hidden-layer membrane potential and spike traces from test data."""
    mem_list, spk_list = [], []
    model.eval()
    
    with torch.no_grad():
        units, times, labels = data_cache.units, data_cache.times, data_cache.labels
        class_list = prms.get("class_list", list(range(20)))
        label_arr = labels if isinstance(labels, np.ndarray) else np.array(labels[:])
        sample_idx = np.where(np.isin(label_arr, class_list))[0]
        
        batch_size = min(prms.get("batch_size", 256), len(sample_idx))
        n_batches = min(n_batches, len(sample_idx) // max(batch_size, 1))
        
        for b in range(n_batches):
            bi = sample_idx[b * batch_size:(b + 1) * batch_size]
            actual_bs = len(bi)
            
            t_arrays = [np.round(times[idx] * (1.0 / prms["time_step"])).astype(np.int64) for idx in bi]
            u_arrays = [units[idx] for idx in bi]
            all_ts = np.concatenate(t_arrays)
            all_us = np.concatenate(u_arrays)
            all_bc = np.repeat(np.arange(actual_bs, dtype=np.int64), [len(a) for a in t_arrays])
            valid = all_ts < prms["nb_steps"]
            all_ts, all_us, all_bc = all_ts[valid], all_us[valid], all_bc[valid]
            
            index_tensor = torch.from_numpy(np.stack([all_bc, all_ts, all_us]))
            values = torch.ones(all_ts.size, dtype=torch.float32)
            x_batch = torch.sparse_coo_tensor(
                index_tensor, values,
                torch.Size([actual_bs, prms["nb_steps"], prms["nb_inputs"]])
            ).to_dense().clamp_(max=1.0).to(DEVICE)
            
            layer_recs = model(0, 0, x_batch)
            layer_out = layer_recs[layer_idx]
            spk, mem = layer_out[0], layer_out[1]
            
            mem_flat = mem.permute(2, 0, 1).reshape(mem.shape[2], -1).cpu().numpy()
            spk_flat = spk.permute(2, 0, 1).reshape(spk.shape[2], -1).cpu().numpy()
            mem_list.append(mem_flat)
            spk_list.append(spk_flat)
    
    mem_all = np.concatenate(mem_list, axis=1)
    spk_all = np.concatenate(spk_list, axis=1)
    return mem_all, spk_all


# ── Load checkpoint and extract traces for seed 101 ──────────────
print("Loading caches...")
train_cache, test_cache = load_default_caches()

WM_SEED = 101
WM_N_BATCHES = 8
WM_STRIDE = 4

wm_pair = build_seeded_pair(
    master_seed=WM_SEED, task_key=TASK_KEY, mem_distribution_family=MEM_DISTRIBUTION_FAMILY)
paths = get_pair_checkpoint_paths(
    master_seed=WM_SEED, run_label=RUN_LABEL, task_key=TASK_KEY,
    mem_distribution_family=MEM_DISTRIBUTION_FAMILY, checkpoint_root=CHECKPOINT_ROOT)

homo_ckpt = torch.load(paths["homogeneous_anchor"], map_location="cpu")
hetero_ckpt = torch.load(paths["heterogeneous_sampled"], map_location="cpu")
wm_pair["hom_model"].load_state_dict(homo_ckpt["model_state_dict"])
wm_pair["hetero_model"].load_state_dict(hetero_ckpt["model_state_dict"])

print(f"Seed {WM_SEED}: Homo acc={homo_ckpt['history']['test_acc'][-1]:.3f}, "
      f"Hetero acc={hetero_ckpt['history']['test_acc'][-1]:.3f}")

print(f"Extracting traces from {WM_N_BATCHES} test batches...")
homo_mem, homo_spk = extract_hidden_traces(
    wm_pair["hom_model"], wm_pair["hom_prms"], test_cache, n_batches=WM_N_BATCHES)
hetero_mem, hetero_spk = extract_hidden_traces(
    wm_pair["hetero_model"], wm_pair["hetero_prms"], test_cache, n_batches=WM_N_BATCHES)

# Temporal downsampling
homo_mem = homo_mem[:, ::WM_STRIDE]
hetero_mem = hetero_mem[:, ::WM_STRIDE]
homo_spk = homo_spk[:, ::WM_STRIDE]
hetero_spk = hetero_spk[:, ::WM_STRIDE]

print(f"  Homo mem:  {homo_mem.shape}  spk: {homo_spk.shape}")
print(f"  Hetero mem: {hetero_mem.shape}  spk: {hetero_spk.shape}")
print(f"  (stride={WM_STRIDE})")

Loading caches...
Pre-loading SHD training data into RAM...
  SHDCache: 8156 samples loaded from shd_train.h5
Pre-loading SHD test data into RAM...
  SHDCache: 2264 samples loaded from shd_test.h5
Seed 101: Homo acc=0.582, Hetero acc=0.662
Extracting traces from 8 test batches...
  Homo mem:  (64, 512000)  spk: (64, 512000)
  Hetero mem: (64, 512000)  spk: (64, 512000)
  (stride=4)


## 4. Filter Active Neurons & Prepare Spike Data

The Zero-M null uses **spike data** (binary), not membrane potentials. We filter to active neurons (firing rate ≥ 1e-9) before W/M computation.

In [5]:
# ── Filter to active neurons ──────────────────────────────────────
homo_fr = homo_spk.mean(axis=1)
hetero_fr = hetero_spk.mean(axis=1)
homo_active = np.where(homo_fr >= 1e-9)[0]
hetero_active = np.where(hetero_fr >= 1e-9)[0]

print(f"Active neurons — Homo: {len(homo_active)}/{64}, Hetero: {len(hetero_active)}/{64}")

# Use SPIKE data for the W/M engine (binary, float64)
ACTIVE_SPK = {
    "homo": homo_spk[homo_active, :].astype(np.float64),
    "hetero": hetero_spk[hetero_active, :].astype(np.float64),
}
MAX_ACTIVE = min(len(homo_active), len(hetero_active))

print(f"Active spike shapes — Homo: {ACTIVE_SPK['homo'].shape}, Hetero: {ACTIVE_SPK['hetero'].shape}")
print(f"Max comparable k: {MAX_ACTIVE}")

Active neurons — Homo: 25/64, Hetero: 33/64
Active spike shapes — Homo: (25, 512000), Hetero: (33, 512000)
Max comparable k: 25


## 5. Observed W/M Sweep (All k, Active-Only Spikes, Robust Estimation)

In [6]:
# ── Configuration ─────────────────────────────────────────────────
SUBSET_SIZES = [k for k in [2, 4, 8, 16, 32] if k <= MAX_ACTIVE]
if MAX_ACTIVE not in SUBSET_SIZES:
    SUBSET_SIZES.append(MAX_ACTIVE)
SUBSET_SAMPLE_SIZE = 50  # subsets per k
LAG = 1
RIDGE = 1e-2

print(f"Subset sizes: {SUBSET_SIZES}")
print(f"Sample size per k: {SUBSET_SAMPLE_SIZE}")
print(f"Lag={LAG}, Ridge={RIDGE}")

# ── Resume from checkpoint ────────────────────────────────────────
_OBS_CHECKPOINT = OUTPUT_DIR / "observed_wm_sweep.csv"
observed_rows = []
completed_pairs = set()  # (net_key, subset_size) already done

if _OBS_CHECKPOINT.exists():
    existing_df = pd.read_csv(_OBS_CHECKPOINT)
    observed_rows = existing_df.to_dict("records")
    for (nk, k), grp in existing_df.groupby(["network", "subset_size"]):
        completed_pairs.add((nk, int(k)))
    print(f"  Resuming: {len(completed_pairs)}/{len(SUBSET_SIZES)*2} (network,k) pairs already done.")

# ── Run observed W/M sweep ────────────────────────────────────────
for net_key, spk in [("homo", ACTIVE_SPK["homo"]), ("hetero", ACTIVE_SPK["hetero"])]:
    nb_hidden = int(spk.shape[0])
    print(f"\n{'='*60}")
    print(f"  {net_key.upper()} — {nb_hidden} active neurons")
    print(f"{'='*60}")
    
    for subset_size in SUBSET_SIZES:
        if (net_key, subset_size) in completed_pairs:
            sub_rows = [r for r in observed_rows
                        if r["network"] == net_key and r["subset_size"] == subset_size]
            ok = sum(1 for r in sub_rows if r["status"] == "ok")
            w_vals = [r["observed_W_bits"] for r in sub_rows if r["status"] == "ok"]
            m_vals = [r["observed_M_bits"] for r in sub_rows if r["status"] == "ok"]
            print(f"  k={subset_size:2d} | (cached) ok={ok:3d} | "
                  f"mean W={np.nanmean(w_vals):.4f} | mean M={np.nanmean(m_vals):.4f}")
            continue
        
        sampled_subsets, total_possible = sample_random_subsets(
            n_neurons=nb_hidden, subset_size=subset_size,
            sample_size=SUBSET_SAMPLE_SIZE,
            seed=_stable_seed(net_key, "subset", subset_size))
        
        k_rows = []
        ok_count = 0
        w_values, m_values = [], []
        
        for subset_draw, subset_idx in enumerate(sampled_subsets, start=1):
            try:
                w_bits, m_bits, n_samp = compute_wm_from_spike_matrix(
                    spk[list(subset_idx), :], lag=LAG, ridge=RIDGE)
                status = "ok"
                ok_count += 1
                w_values.append(float(w_bits))
                m_values.append(float(m_bits))
            except Exception as exc:
                w_bits, m_bits, n_samp = np.nan, np.nan, 0
                status = f"error: {exc}"
            
            row_dict = {
                "network": net_key, "subset_size": int(subset_size),
                "subset_draw": int(subset_draw),
                "subset_indices": ",".join(map(str, subset_idx)),
                "total_possible_subsets": int(total_possible),
                "observed_W_bits": float(w_bits) if np.isfinite(w_bits) else np.nan,
                "observed_M_bits": float(m_bits) if np.isfinite(m_bits) else np.nan,
                "status": status,
            }
            k_rows.append(row_dict)
            observed_rows.append(row_dict)
        
        print(f"  k={subset_size:2d} | sampled {len(sampled_subsets):3d}/{total_possible} | "
              f"ok={ok_count:3d} | mean W={np.nanmean(w_values):.4f} | mean M={np.nanmean(m_values):.4f}")
        
        # ── Incremental save after each k ─────────────────────────
        pd.DataFrame(observed_rows).to_csv(_OBS_CHECKPOINT, index=False)

observed_df = pd.DataFrame.from_records(observed_rows)
print(f"\n✓ Observed W/M sweep saved to {_OBS_CHECKPOINT}")

# Summary
obs_summary = (observed_df.groupby(["network", "subset_size"], as_index=False)
    .agg(observed_W_mean=("observed_W_bits", "mean"),
         observed_M_mean=("observed_M_bits", "mean"),
         ok_count=("status", lambda x: (x == "ok").sum())))
display(obs_summary)

Subset sizes: [2, 4, 8, 16, 25]
Sample size per k: 50
Lag=1, Ridge=0.01
  Resuming: 10/10 (network,k) pairs already done.

  HOMO — 25 active neurons
  k= 2 | (cached) ok= 50 | mean W=2.8426 | mean M=0.0023
  k= 4 | (cached) ok= 50 | mean W=5.2215 | mean M=0.0275
  k= 8 | (cached) ok= 50 | mean W=9.5727 | mean M=0.1183
  k=16 | (cached) ok= 50 | mean W=16.2009 | mean M=0.4007
  k=25 | (cached) ok=  1 | mean W=22.0897 | mean M=0.7927

  HETERO — 33 active neurons
  k= 2 | (cached) ok= 50 | mean W=3.1360 | mean M=0.0029
  k= 4 | (cached) ok= 50 | mean W=5.8766 | mean M=0.0358
  k= 8 | (cached) ok= 50 | mean W=10.5933 | mean M=0.1753
  k=16 | (cached) ok= 50 | mean W=16.5305 | mean M=0.6132
  k=25 | (cached) ok= 50 | mean W=18.7753 | mean M=1.2896

✓ Observed W/M sweep saved to C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files\Checkpoints\SeededRuns\seeded_run_1_64n\4class_lognormal\zero_m_zscores\observed_wm_sweep.csv


,network,subset_size,observed_W_mean,observed_M_mean,ok_count
0,hetero,2,3.136033,0.002857,50
1,hetero,4,5.876573,0.035811,50
2,hetero,8,10.593329,0.175344,50
3,hetero,16,16.530509,0.613175,50
4,hetero,25,18.775255,1.289648,50
5,homo,2,2.842614,0.002285,50
6,homo,4,5.221451,0.027502,50
7,homo,8,9.572740,0.118307,50
8,homo,16,16.200888,0.400701,50
9,homo,25,22.089716,0.792663,1


## 6. Zero-M Null Z-Scoring

For each observed subset, we:
1. Compute TDMI = W + M
2. Build a VAR(1) model with the **same TDMI** but **M=0** by construction
3. Generate N_NULL samples from the null model
4. Compare observed M to null M distribution → Z-score + p-values

In [7]:
# ── Zero-M Null Z-Scoring (with incremental checkpointing) ────────
_ZSCORE_CHECKPOINT = OUTPUT_DIR / "zero_m_zscores.csv"

# Resume from checkpoint
records = []
completed_z_pairs = set()
if _ZSCORE_CHECKPOINT.exists():
    existing_z = pd.read_csv(_ZSCORE_CHECKPOINT)
    records = existing_z.to_dict("records")
    for (nk, k), grp in existing_z.groupby(["network", "subset_size"]):
        completed_z_pairs.add((nk, int(k)))
    print(f"  Resuming z-scores: {len(completed_z_pairs)} (network,k) pairs already done.")

for (net_key, subset_size), grp in observed_df.groupby(["network", "subset_size"]):
    if (net_key, int(subset_size)) in completed_z_pairs:
        sub = [r for r in records if r["network"] == net_key and r["subset_size"] == int(subset_size)]
        z_vals = [r["M_z_tdmi_null"] for r in sub if np.isfinite(r["M_z_tdmi_null"])]
        p_vals = [r["M_p_two_sided"] for r in sub if np.isfinite(r["M_p_two_sided"])]
        sig_count = sum(1 for p in p_vals if p < 0.05)
        print(f"\nScoring {net_key} / k={int(subset_size)}  (cached)")
        print(f"  Scored={len(sub)} | mean z={np.nanmean(z_vals):.3f} | "
              f"p<0.05: {sig_count}/{len(p_vals)}")
        continue
    
    print(f"\nScoring {net_key} / k={int(subset_size)}")
    
    k_records = []
    for _, row in grp.iterrows():
        obs_m = float(row["observed_M_bits"])
        obs_w = float(row["observed_W_bits"])
        
        if not (np.isfinite(obs_m) and np.isfinite(obs_w)):
            continue
        
        tdmi_target = float(obs_m + obs_w)
        
        t0 = time_module.perf_counter()
        null_dist = build_numit_null_distribution(
            nvar=int(subset_size),
            tdmi_target_bits=tdmi_target,
            n_null=N_NULL,
            seed=_stable_seed(net_key, "null", subset_size, row["subset_draw"]),
            ridge=RIDGE,
            max_attempts=MAX_ATTEMPTS,
        )
        elapsed = time_module.perf_counter() - t0
        
        score = score_observed_vs_null_m(obs_m, null_dist["null_M_values"])
        
        rec = {
            "network": net_key,
            "subset_size": int(subset_size),
            "subset_draw": int(row["subset_draw"]),
            "observed_M_bits": obs_m,
            "observed_W_bits": obs_w,
            "observed_TDMI_bits": tdmi_target,
            "null_M_mean": score["null_mean"],
            "null_M_std": score["null_std"],
            "M_z_tdmi_null": score["z"],
            "M_p_upper": score["p_upper"],
            "M_p_lower": score["p_lower"],
            "M_p_two_sided": score["p_two_sided"],
            "n_null_valid": null_dist["n_null_valid"],
            "n_attempts": null_dist["n_attempts"],
            "null_elapsed_s": elapsed,
        }
        k_records.append(rec)
        records.append(rec)
    
    # Print summary for this k
    z_vals = [r["M_z_tdmi_null"] for r in k_records if np.isfinite(r["M_z_tdmi_null"])]
    p_vals = [r["M_p_two_sided"] for r in k_records if np.isfinite(r["M_p_two_sided"])]
    sig_count = sum(1 for p in p_vals if p < 0.05)
    print(f"  Scored={len(k_records)} | mean z={np.nanmean(z_vals):.3f} | "
          f"p<0.05: {sig_count}/{len(p_vals)} | "
          f"mean null_attempts={np.mean([r['n_attempts'] for r in k_records]):.0f}")
    
    # ── Incremental save after each k ─────────────────────────────
    pd.DataFrame(records).to_csv(_ZSCORE_CHECKPOINT, index=False)

scores_df = pd.DataFrame.from_records(records)
scores_df = scores_df.sort_values(["network", "subset_size", "subset_draw"]).reset_index(drop=True)
scores_df.to_csv(_ZSCORE_CHECKPOINT, index=False)
print(f"\n✓ Zero-M Z-scores saved to {_ZSCORE_CHECKPOINT}")

# Summary table
score_summary = (scores_df.groupby(["network", "subset_size"], as_index=False)
    .agg(observed_M_mean=("observed_M_bits", "mean"),
         M_z_mean=("M_z_tdmi_null", "mean"),
         M_z_median=("M_z_tdmi_null", "median"),
         M_p_upper_mean=("M_p_upper", "mean"),
         M_p_lower_mean=("M_p_lower", "mean"),
         p_sig_frac=("M_p_two_sided", lambda x: (x < 0.05).mean()),
         n_scored=("subset_draw", "count")))
display(score_summary)

  Resuming z-scores: 2 (network,k) pairs already done.

Scoring hetero / k=2  (cached)
  Scored=50 | mean z=-1.096 | p<0.05: 0/50

Scoring hetero / k=4  (cached)
  Scored=50 | mean z=-1.941 | p<0.05: 0/50

Scoring hetero / k=8
  Scored=50 | mean z=-3.125 | p<0.05: 0/50 | mean null_attempts=24

Scoring hetero / k=16
  Scored=50 | mean z=-9.188 | p<0.05: 0/50 | mean null_attempts=21

Scoring hetero / k=25
  Scored=50 | mean z=-21.605 | p<0.05: 0/50 | mean null_attempts=20

Scoring homo / k=2
  Scored=50 | mean z=-1.176 | p<0.05: 0/50 | mean null_attempts=20

Scoring homo / k=4
  Scored=50 | mean z=-2.214 | p<0.05: 0/50 | mean null_attempts=20

Scoring homo / k=8
  Scored=50 | mean z=-3.716 | p<0.05: 0/50 | mean null_attempts=21

Scoring homo / k=16
  Scored=50 | mean z=-11.390 | p<0.05: 0/50 | mean null_attempts=20

Scoring homo / k=25
  Scored=1 | mean z=-25.517 | p<0.05: 0/1 | mean null_attempts=20

✓ Zero-M Z-scores saved to C:\Users\Priya\Desktop\research project (SNN Info Theory)\Pr

,network,subset_size,observed_M_mean,M_z_mean,M_z_median,M_p_upper_mean,M_p_lower_mean,p_sig_frac,n_scored
0,hetero,2,0.002857,-1.096306,-1.069349,0.984762,0.062857,0.0,50
1,hetero,4,0.035811,-1.941374,-1.885334,1.000000,0.047619,0.0,50
2,hetero,8,0.175344,-3.125405,-2.725970,1.000000,0.047619,0.0,50
3,hetero,16,0.613175,-9.187896,-8.981485,1.000000,0.047619,0.0,50
4,hetero,25,1.289648,-21.605233,-21.486777,1.000000,0.047619,0.0,50
5,homo,2,0.002285,-1.176121,-1.196368,0.991429,0.056190,0.0,50
6,homo,4,0.027502,-2.213535,-2.093797,1.000000,0.047619,0.0,50
7,homo,8,0.118307,-3.716254,-3.453085,1.000000,0.047619,0.0,50
8,homo,16,0.400701,-11.390170,-11.604825,1.000000,0.047619,0.0,50
9,homo,25,0.792663,-25.516540,-25.516540,1.000000,0.047619,0.0,1


## 7. Summary Plots

In [ ]:
# ── Prepare data ──────────────────────────────────────────────────
homo_obs = observed_df[observed_df["network"] == "homo"]
hetero_obs = observed_df[observed_df["network"] == "hetero"]
homo_scores = scores_df[scores_df["network"] == "homo"]
hetero_scores = scores_df[scores_df["network"] == "hetero"]
subset_sizes_plot = sorted(scores_df["subset_size"].unique())

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# (0,0) W-information vs k
ax = axes[0, 0]
for net, df, color, marker, label in [
    ("homo", homo_obs, "#2196F3", "o", "Homogeneous"),
    ("hetero", hetero_obs, "#FF5722", "s", "Heterogeneous")]:
    grp = df.groupby("subset_size")["observed_W_bits"]
    means = grp.mean()
    sems = grp.sem()
    ks = means.index.values
    ax.errorbar(ks, means, yerr=sems, color=color, marker=marker, ms=8, lw=2,
                capsize=5, label=label)
ax.set_xlabel("Subset size k"); ax.set_ylabel("W-information (bits)")
ax.set_title(f"W-Information — Seed {WM_SEED}", fontweight="bold")
ax.legend(); ax.grid(True, alpha=0.3)

# (0,1) M-information vs k
ax = axes[0, 1]
for net, df, color, marker, label in [
    ("homo", homo_obs, "#2196F3", "o", "Homogeneous"),
    ("hetero", hetero_obs, "#FF5722", "s", "Heterogeneous")]:
    grp = df.groupby("subset_size")["observed_M_bits"]
    means = grp.mean()
    sems = grp.sem()
    ks = means.index.values
    ax.errorbar(ks, means, yerr=sems, color=color, marker=marker, ms=8, lw=2,
                capsize=5, label=label)
ax.set_xlabel("Subset size k"); ax.set_ylabel("M-information (bits)")
ax.set_title(f"M-Information — Seed {WM_SEED}", fontweight="bold")
ax.legend(); ax.grid(True, alpha=0.3)

# (1,0) M Z-score boxplots
ax = axes[1, 0]
positions_homo = np.arange(len(subset_sizes_plot)) * 2 - 0.35
positions_hetero = np.arange(len(subset_sizes_plot)) * 2 + 0.35

for pos, color, label, df_sub in [
    (positions_homo, "#2196F3", "Homo", homo_scores),
    (positions_hetero, "#FF5722", "Hetero", hetero_scores)]:
    box_data = []
    for k in subset_sizes_plot:
        vals = df_sub[df_sub["subset_size"] == k]["M_z_tdmi_null"].dropna().values
        box_data.append(vals if len(vals) > 0 else [np.nan])
    bp = ax.boxplot(box_data, positions=pos, widths=0.55, patch_artist=True,
                     showfliers=True, manage_ticks=False)
    for patch in bp["boxes"]:
        patch.set_facecolor(color)
        patch.set_alpha(0.5)

ax.axhline(y=0, color="black", lw=0.5)
ax.set_xticks(np.arange(len(subset_sizes_plot)) * 2)
ax.set_xticklabels([str(k) for k in subset_sizes_plot])
ax.set_xlabel("Subset size k"); ax.set_ylabel("M Z-score (Zero-M null)")
ax.set_title(f"M-Info Z-Scores — Seed {WM_SEED}", fontweight="bold")
ax.legend([plt.Rectangle((0,0),1,1,fc="#2196F3",alpha=0.5),
           plt.Rectangle((0,0),1,1,fc="#FF5722",alpha=0.5)],
          ["Homogeneous", "Heterogeneous"])
ax.grid(True, alpha=0.3, axis="y")

# (1,1) ΔM Z-score bar chart
ax = axes[1, 1]
delta_z = []
for k in subset_sizes_plot:
    hz = hetero_scores[hetero_scores["subset_size"] == k]["M_z_tdmi_null"].mean()
    hmz = homo_scores[homo_scores["subset_size"] == k]["M_z_tdmi_null"].mean()
    delta_z.append(hz - hmz)
colors = ["#4CAF50" if d > 0 else "#F44336" for d in delta_z]
ax.bar(range(len(subset_sizes_plot)), delta_z, color=colors, edgecolor="white", lw=0.5)
ax.set_xticks(range(len(subset_sizes_plot)))
ax.set_xticklabels([str(k) for k in subset_sizes_plot])
ax.set_xlabel("Subset size k"); ax.set_ylabel("Δ Z-score (hetero − homo)")
ax.set_title("M Z-Score Gain from Heterogeneity", fontweight="bold")
ax.axhline(y=0, color="black", lw=0.8); ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plot_path = OUTPUT_DIR / "zero_m_zscores_plot.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Plot saved to {plot_path}")

# ── Summary table ──────────────────────────────────────────────────
print(f"\n{'='*105}")
print(f"  Zero-M Z-Score Summary — Seed {WM_SEED}")
print(f"{'='*105}")
print(f"{'k':>3}  {'ΔW':>8}  {'ΔM':>8}  {'z_M_homo':>10}  {'z_M_hetero':>10}  "
      f"{'p_sig_h':>7}  {'p_sig_t':>7}  "
      f"{'p_up_h':>7}  {'p_up_t':>7}  {'p_lo_h':>7}  {'p_lo_t':>7}")
print("-" * 105)
for k in subset_sizes_plot:
    h_w = homo_obs[homo_obs["subset_size"]==k]["observed_W_bits"].mean()
    ht_w = hetero_obs[hetero_obs["subset_size"]==k]["observed_W_bits"].mean()
    h_m = homo_obs[homo_obs["subset_size"]==k]["observed_M_bits"].mean()
    ht_m = hetero_obs[hetero_obs["subset_size"]==k]["observed_M_bits"].mean()
    h_z = homo_scores[homo_scores["subset_size"]==k]["M_z_tdmi_null"].mean()
    ht_z = hetero_scores[hetero_scores["subset_size"]==k]["M_z_tdmi_null"].mean()
    h_psig = (homo_scores[homo_scores["subset_size"]==k]["M_p_two_sided"] < 0.05).mean()
    ht_psig = (hetero_scores[hetero_scores["subset_size"]==k]["M_p_two_sided"] < 0.05).mean()
    h_pup = homo_scores[homo_scores["subset_size"]==k]["M_p_upper"].mean()
    ht_pup = hetero_scores[hetero_scores["subset_size"]==k]["M_p_upper"].mean()
    h_plo = homo_scores[homo_scores["subset_size"]==k]["M_p_lower"].mean()
    ht_plo = hetero_scores[hetero_scores["subset_size"]==k]["M_p_lower"].mean()
    
    dw = ht_w - h_w
    dm = ht_m - h_m
    print(f"{k:>3}  {dw:>+8.4f}  {dm:>+8.4f}  {h_z:>10.2f}  {ht_z:>10.2f}  "
          f"{h_psig:>7.2f}  {ht_psig:>7.2f}  "
          f"{h_pup:>7.3f}  {ht_pup:>7.3f}  {h_plo:>7.3f}  {ht_plo:>7.3f}")

print(f"\n  Legend: h=homo, t=hetero (target)")
print(f"  p_sig  = fraction of draws with p_two_sided < 0.05")
print(f"  p_up   = mean p_upper  (P(null M ≥ obs M))  →  significant if p_up is small")
print(f"  p_lo   = mean p_lower  (P(null M ≤ obs M))  →  significant if p_lo is small")


  Zero-M Z-Score Summary — Seed 101
  k        ΔW        ΔM    z_M_homo  z_M_hetero  p_sig_h  p_sig_t   p_up_h   p_up_t   p_lo_h   p_lo_t
---------------------------------------------------------------------------------------------------------


NameError: name 'subset_sizes_plot' is not defined

## 8. Key Findings

*To be completed after execution.*

**Expected advantages over time-shuffle null:**
1. Zero-M null properly models "no synergy" — M=0 by construction
2. Robust estimation avoids NaN at higher k via ridge regularization + multi-strategy fallback
3. Z-scores should be in a realistic range (typically 0–10, not 10⁶)
4. p-values are meaningful and can be compared to α=0.05

**Comparison to time-shuffle approach:**
- Time-shuffle: Z-scores ~10⁶ (meaningless), k≥8 fails entirely
- Zero-M: Expected Z-scores 0–5 range, should work at all k

**Key metrics to check:**
- Fraction of subsets with p<0.05 for hetero vs homo
- ΔZ (hetero−homo) trend with k
- Whether M-information is significantly above Zero-M null